<a href="https://colab.research.google.com/github/MatthewFaiello/MLsessions/blob/main/ISEA_Week5_ML_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Your Task: Predict Student Success

The purpose of this HW is to get you hands on with real data trying out the modelling techniques we talked about.

You are free to use gen-ai with this project to help with the coding (of course, you don't have to!). [Hands on Machine Learning](https://www.oreilly.com/library/view/hands-on-machine-learning/9781098125967/) is also a great resource.

Your code needs to run, but I want you to focus less on the specifics of the code and more on understanding the component steps that go into building and validating a model. Creating code is now pretty easy, creating a "good" model is hard.

For this exercise we will use open data on student dropout from Portugal. Full documentation is available [here](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)

M.V.Martins, D. Tolledo, J. Machado, L. M.T. Baptista, V.Realinho. (2021) "Early prediction of student’s performance in higher education: a case study" Trends and Applications in Information Systems and Technologies, vol.1, in Advances in Intelligent Systems and Computing series. Springer. DOI: 10.1007/978-3-030-72657-7_16

You will turn in on the class website a google slide deck that has:
1. A cover slide contianing your name (and all group member names if working together) and a link to your colab (**Create slide 1 now**)
2. 3 slides answering the questions below - they are clearly indicated as you go through the colab notebook.


# Get the data

Here I provide some code to get the data for you

In [243]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

import pandas as pd

from sklearn.model_selection import train_test_split

import numpy as np

In [244]:
# fetch dataset
predict_students_dropout_and_academic_success = fetch_ucirepo(id=697)

# data (as pandas dataframes)
X = predict_students_dropout_and_academic_success.data.features
y = predict_students_dropout_and_academic_success.data.targets
df = df = X.join(y)

# metadata
print(predict_students_dropout_and_academic_success.metadata)

# variable information
print(predict_students_dropout_and_academic_success.variables)

{'uci_id': 697, 'name': "Predict Students' Dropout and Academic Success", 'repository_url': 'https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success', 'data_url': 'https://archive.ics.uci.edu/static/public/697/data.csv', 'abstract': "A dataset created from a higher education institution (acquired from several disjoint databases) related to students enrolled in different undergraduate degrees, such as agronomy, design, education, nursing, journalism, management, social service, and technologies.\nThe dataset includes information known at the time of student enrollment (academic path, demographics, and social-economic factors) and the students' academic performance at the end of the first and second semesters. \nThe data is used to build classification models to predict students' dropout and academic sucess. The problem is formulated as a three category classification task, in which there is a strong imbalance towards one of the classes.", 'area': 'Social Sc

# 1 Data Checking

- Look at your outcome variable - any cases to exclude?
- Determine the base-rate accuracy for a naive model
- Create Test and Training Sets
- Look at distributions of x variables, look up meta data, decide which to include

At the end of this section you should have
`x_train`, `x_text`, `y_train`, `y_test`
And an estimate of the base rate accuracy.

In [245]:
# check target distribution
pd.DataFrame({"count": df["Target"].value_counts(dropna=False),
              "percent": df["Target"].value_counts(normalize=True, dropna=False).mul(100).round(2)})

# recode so target is either "Graduate" or "Other"
df["Target_"] = df["Target"].replace({"Dropout": 0, "Enrolled": 0, "Graduate": 1})

# target base-rate
pd.DataFrame({"count": df["Target_"].value_counts(dropna=False),
              "percent": df["Target_"].value_counts(normalize=True, dropna=False).mul(100).round(2)})

# check features
metaData = predict_students_dropout_and_academic_success.variables; metaData
num = df.select_dtypes(include="number")
pd.DataFrame({"mean": num.mean(),
              "median": num.median(),
              "mode": num.mode(dropna=True).iloc[0]})

# select features
range_cols = df.iloc[:, np.r_[4, 6, 12:32, 37]].columns
df_subset = df.loc[:, range_cols]; df_subset

# stratified test and train sets
X = df_subset.drop(columns=["Target_"])
y = df_subset["Target_"]

# split (stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)
# check stratification
pd.DataFrame({"y_train": y_train.value_counts(normalize=True, dropna=False).mul(100).round(2),
              "y_test": y_test.value_counts(normalize=True, dropna=False).mul(100).round(2)})

/tmp/ipython-input-303/1281890171.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Target_"] = df["Target"].replace({"Dropout": 0, "Enrolled": 0, "Graduate": 1})


,y_train,y_test
Target_,,
0,50.07,50.06
1,49.93,49.94


# 2 Train a Model
* Pick one of the models we discussed today and train it.
* Report its accuracy and print a confusion matrix.
   * How much better is your model than the base rate?
   * How does accuracy on the train set compare to accuracy on the test set?
   * **Report Slide 2: Include an image of the confusion matrix, the base rate accuracy, train-set accuracy and test-set accuracy**

In [246]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    accuracy_score
)

# -----------------------------------------------------------------------------
# 0) Shared CV strategy (both models should use the SAME CV scheme)
# -----------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def print_best_cv_summary(gs, label):
    best_idx = gs.best_index_
    cv_mean = gs.cv_results_["mean_test_score"][best_idx]
    cv_std  = gs.cv_results_["std_test_score"][best_idx]
    print(f"\n[{label}] Best params: {gs.best_params_}")
    print(f"[{label}] CV best balanced acc: {gs.best_score_:.4f}")
    print(f"[{label}] CV balanced acc (mean ± std): {cv_mean:.4f} ± {cv_std:.4f}")

# -----------------------------------------------------------------------------
# Helper: majority-class (base rate) accuracy
#   - "What accuracy do I get if I always predict the most frequent class?"
# -----------------------------------------------------------------------------
def majority_baseline_accuracy(y):
    if hasattr(y, "value_counts"):
        return y.value_counts(normalize=True).max()
    vals, counts = np.unique(np.asarray(y), return_counts=True)
    return counts.max() / counts.sum()

# -----------------------------------------------------------------------------
# Helper: evaluate model on train/test + include improvement over baseline
# -----------------------------------------------------------------------------
def evaluate_model(model, X_train, y_train, X_test, y_test, labels, baseline_train, baseline_test):
    pred_train = model.predict(X_train)
    pred_test  = model.predict(X_test)

    train_acc = accuracy_score(y_train, pred_train)
    test_acc  = accuracy_score(y_test, pred_test)

    # Improvement over baseline (accuracy points, not %)
    # Example: test_acc=0.78 baseline=0.70 => improvement=0.08 (8 percentage points)
    train_improve = train_acc - baseline_train
    test_improve  = test_acc  - baseline_test

    metrics = {
        "train_accuracy": train_acc,
        "test_accuracy": test_acc,
        "train_balanced_accuracy": balanced_accuracy_score(y_train, pred_train),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, pred_test),

        # New: baseline + improvement (so comparison table is self-contained)
        "train_majority_baseline": baseline_train,
        "test_majority_baseline": baseline_test,
        "train_improvement_over_baseline": train_improve,
        "test_improvement_over_baseline": test_improve,
    }

    cm_train = confusion_matrix(y_train, pred_train, labels=labels)
    cm_test  = confusion_matrix(y_test, pred_test, labels=labels)

    return metrics, cm_train, cm_test, pred_train, pred_test

def cm_df(cm, labels):
    return pd.DataFrame(
        cm,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels],
    )

# -----------------------------------------------------------------------------
# Fixed label order for both models (so matrices align)
# -----------------------------------------------------------------------------
labels = np.unique(np.concatenate([np.asarray(y_train), np.asarray(y_test)]))

# -----------------------------------------------------------------------------
# Compute baselines ONCE (train + test)
# -----------------------------------------------------------------------------
baseline_train = majority_baseline_accuracy(y_train)
baseline_test  = majority_baseline_accuracy(y_test)

print("\n==================== BASELINES (Majority-class Accuracy) ====================")
print(f"Train majority baseline: {baseline_train:.4f} ({baseline_train*100:.2f}%)")
print(f"Test  majority baseline: {baseline_test:.4f} ({baseline_test*100:.2f}%)")


==================== BASELINES (Majority-class Accuracy) ====================
Train majority baseline: 0.5007 (50.07%)
Test  majority baseline: 0.5006 (50.06%)


In [247]:
# =============================================================================
# 1) L1 LOGISTIC REGRESSION
# =============================================================================
pipe_lr = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=10000,
        random_state=42
    )),
])

param_grid_lr = {
    "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "clf__class_weight": [None, "balanced"],
}

gs_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1,
    refit=True
)

gs_lr.fit(X_train, y_train)
print_best_cv_summary(gs_lr, "L1 Logistic")
best_lr = gs_lr.best_estimator_

lr_metrics, lr_cm_train, lr_cm_test, lr_pred_train, lr_pred_test = evaluate_model(
    best_lr, X_train, y_train, X_test, y_test, labels, baseline_train, baseline_test
)

print("\n[L1 Logistic] Train/Test metrics (incl. baseline improvements):")
for k, v in lr_metrics.items():
    # show improvements in percentage points for readability
    if "improvement_over_baseline" in k:
        print(f"  {k:33s}: {v:+.4f} ({v*100:+.2f} pp)")
    else:
        print(f"  {k:33s}: {v:.4f}")

print("\n[L1 Logistic] Confusion matrix (test):")
print(cm_df(lr_cm_test, labels))


[L1 Logistic] Best params: {'clf__C': 1, 'clf__class_weight': None}
[L1 Logistic] CV best balanced acc: 0.8506
[L1 Logistic] CV balanced acc (mean ± std): 0.8506 ± 0.0154

[L1 Logistic] Train/Test metrics (incl. baseline improvements):
  train_accuracy                   : 0.8562
  test_accuracy                    : 0.8260
  train_balanced_accuracy          : 0.8562
  test_balanced_accuracy           : 0.8260
  train_majority_baseline          : 0.5007
  test_majority_baseline           : 0.5006
  train_improvement_over_baseline  : +0.3555 (+35.55 pp)
  test_improvement_over_baseline   : +0.3254 (+32.54 pp)

[L1 Logistic] Confusion matrix (test):
        pred_0  pred_1
true_0     343     100
true_1      54     388


# 3 Train a Different Model
* Repeat all the steps in 2, but use a different model
* In addition, compare the accuracy of 1 and 2
* **Report Slide 3: Model 2 confusion matrix, train-set accuracy and test-set accuracy. Comparison Model 1 and Model 2 accuracy**

In [248]:
# =============================================================================
# 2) DECISION TREE
# =============================================================================
pipe_dt = Pipeline(steps=[
    ("clf", DecisionTreeClassifier(random_state=42))
])

param_grid_dt = {
    "clf__criterion": ["gini", "entropy", "log_loss"],
    "clf__max_depth": [None, 3, 5, 10, 20],
    "clf__min_samples_split": [2, 5, 10, 20],
    "clf__min_samples_leaf": [1, 2, 5, 10],
    "clf__ccp_alpha": [0.0, 0.001, 0.01],
    "clf__class_weight": [None, "balanced"],
}

gs_dt = GridSearchCV(
    estimator=pipe_dt,
    param_grid=param_grid_dt,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1,
    refit=True
)

gs_dt.fit(X_train, y_train)
print_best_cv_summary(gs_dt, "Decision Tree")
best_dt = gs_dt.best_estimator_

dt_metrics, dt_cm_train, dt_cm_test, dt_pred_train, dt_pred_test = evaluate_model(
    best_dt, X_train, y_train, X_test, y_test, labels, baseline_train, baseline_test
)

print("\n[Decision Tree] Train/Test metrics (incl. baseline improvements):")
for k, v in dt_metrics.items():
    if "improvement_over_baseline" in k:
        print(f"  {k:33s}: {v:+.4f} ({v*100:+.2f} pp)")
    else:
        print(f"  {k:33s}: {v:.4f}")

print("\n[Decision Tree] Confusion matrix (test):")
print(cm_df(dt_cm_test, labels))


[Decision Tree] Best params: {'clf__ccp_alpha': 0.001, 'clf__class_weight': None, 'clf__criterion': 'gini', 'clf__max_depth': 10, 'clf__min_samples_leaf': 5, 'clf__min_samples_split': 2}
[Decision Tree] CV best balanced acc: 0.8381
[Decision Tree] CV balanced acc (mean ± std): 0.8381 ± 0.0209

[Decision Tree] Train/Test metrics (incl. baseline improvements):
  train_accuracy                   : 0.8712
  test_accuracy                    : 0.8056
  train_balanced_accuracy          : 0.8712
  test_balanced_accuracy           : 0.8057
  train_majority_baseline          : 0.5007
  test_majority_baseline           : 0.5006
  train_improvement_over_baseline  : +0.3704 (+37.04 pp)
  test_improvement_over_baseline   : +0.3051 (+30.51 pp)

[Decision Tree] Confusion matrix (test):
        pred_0  pred_1
true_0     339     104
true_1      68     374


In [249]:
# =============================================================================
# 3) HEAD-TO-HEAD COMPARISON
# =============================================================================
comparison = pd.DataFrame(
    [lr_metrics, dt_metrics],
    index=["L1 Logistic", "Decision Tree"]
)

print("\n==================== COMPARISON: METRICS ====================")
# Format improvements as percentage points; others as decimals
pretty = comparison.copy()
for col in pretty.columns:
    if "improvement_over_baseline" in col:
        pretty[col] = pretty[col].map(lambda x: f"{x*100:+.2f} pp")
    else:
        pretty[col] = pretty[col].map(lambda x: f"{x:.4f}")
print(pretty)

delta = comparison.loc["Decision Tree"] - comparison.loc["L1 Logistic"]
print("\n==================== Δ (Tree - Logistic) ====================")
# Also show improvement deltas in pp
for k, v in delta.items():
    if "improvement_over_baseline" in k:
        print(f"{k:33s}: {v*100:+.2f} pp")
    else:
        print(f"{k:33s}: {v:+.4f}")

print("\n==================== COMPARISON: CONFUSION MATRICES (TEST) ====================")
print("\n-- L1 Logistic (test) --")
print(cm_df(lr_cm_test, labels))
print("\n-- Decision Tree (test) --")
print(cm_df(dt_cm_test, labels))
print("\n-- Difference (test): Tree - Logistic --")
print(cm_df(dt_cm_test - lr_cm_test, labels))


==================== COMPARISON: METRICS ====================
              train_accuracy test_accuracy train_balanced_accuracy  \
L1 Logistic           0.8562        0.8260                  0.8562   
Decision Tree         0.8712        0.8056                  0.8712   

              test_balanced_accuracy train_majority_baseline  \
L1 Logistic                   0.8260                  0.5007   
Decision Tree                 0.8057                  0.5007   

              test_majority_baseline train_improvement_over_baseline  \
L1 Logistic                   0.5006                       +35.55 pp   
Decision Tree                 0.5006                       +37.04 pp   

              test_improvement_over_baseline  
L1 Logistic                        +32.54 pp  
Decision Tree                      +30.51 pp  

==================== Δ (Tree - Logistic) ====================
train_accuracy                   : +0.0150
test_accuracy                    : -0.0203
train_balanced_accuracy   

# 4 Reflection
* **Type responses on Slide 4**
* Contextualizing accuracy - think about different use cases for your model, which ones would you feel its accurate enough to use for? I only asked you to look at overall accuracy, is that good enough?
* Contextualizing features - think about these same use cases, are the prediction features you included appropriate for these uses?
* Generalizability - again thinking about your features, could you use this model in other educational contexts? How hard would it be to get that same data? Are there issues with it generalizing over time and location?

# 5 Extra Credit
* Consider ensembling your two models. Does that perform better?
* Check accuracy for different subgroups